In [1]:
import numpy as np

from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import SparsePauliOp

from qiskit_aer.primitives import EstimatorV2 as Estimator

from qiskit_machine_learning.neural_networks import EstimatorQNN
from qiskit_machine_learning.connectors import TorchConnector

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler

In [2]:
import pandas as pd

n_samples = 2000  # subset — full 155K rows is impractical for QNN

# Shuffle before sampling so we get a mix of fire/no-fire rows
df_full = pd.read_csv("features.csv").sample(n=n_samples, random_state=42).reset_index(drop=True)
df_x    = df_full[["month_sin","month_cos","year","avg_tmax_c","temp_range",
                    "tot_prcp_mm","dryness_3m_avg","latitude","longitude"]]

x = df_x.values.astype(float)
y = df_full["target"].values.astype(float)

# split and scale
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.1, random_state=0)

x_scaler = StandardScaler()
x_train_s = x_scaler.fit_transform(x_train)
x_test_s  = x_scaler.transform(x_test)

angle_scaler = MinMaxScaler(feature_range=(0, 2 * np.pi))
x_train_a = angle_scaler.fit_transform(x_train_s)
x_test_a  = angle_scaler.transform(x_test_s)

y_scaler = MinMaxScaler(feature_range=(-1, 1))
y_train_s = y_scaler.fit_transform(y_train.reshape(-1, 1)).ravel()
y_test_s  = y_scaler.transform(y_test.reshape(-1, 1)).ravel()

print(f"Train: {x_train_a.shape}, Test: {x_test_a.shape}")
print(f"y_train range: [{y_train_s.min():.3f}, {y_train_s.max():.3f}]")

Train: (1800, 9), Test: (200, 9)
y_train range: [-1.000, 1.000]


In [ ]:
n_features = x_train_a.shape[1]
n_qubits = n_features

x_params = ParameterVector("x", n_features)
reps = 3  # increase to 2 or 3 once pipeline is confirmed working
theta_params = ParameterVector("θ", length=n_qubits * 2 * reps)

qc = QuantumCircuit(n_qubits)
for i in range(n_qubits):
    qc.ry(x_params[i], i)

t = 0
for r in range(reps):
    for q in range(n_qubits):
        qc.ry(theta_params[t], q); t += 1
        qc.rz(theta_params[t], q); t += 1

    for q in range(n_qubits - 1):
        qc.cx(q, q + 1)
    if n_qubits > 2:
        qc.cx(n_qubits - 1, 0)

qc.decompose().draw("text")

global phase: (-0.5)*θ[1] - 0.5*θ[3] - 0.5*θ[5] - 0.5*θ[7] - 0.5*θ[9] - 0.5*θ[11] - 0.5*θ[13] - 0.5*θ[15] - 0.5*θ[17]
     ┌─────────────┐┌─────────────┐ ┌─────────┐                               »
q_0: ┤ R(x[0],π/2) ├┤ R(θ[0],π/2) ├─┤ P(θ[1]) ├───■───────────────────────────»
     ├─────────────┤├─────────────┤ ├─────────┤ ┌─┴─┐                         »
q_1: ┤ R(x[1],π/2) ├┤ R(θ[2],π/2) ├─┤ P(θ[3]) ├─┤ X ├──■──────────────────────»
     ├─────────────┤├─────────────┤ ├─────────┤ └───┘┌─┴─┐                    »
q_2: ┤ R(x[2],π/2) ├┤ R(θ[4],π/2) ├─┤ P(θ[5]) ├──────┤ X ├──■─────────────────»
     ├─────────────┤├─────────────┤ ├─────────┤      └───┘┌─┴─┐               »
q_3: ┤ R(x[3],π/2) ├┤ R(θ[6],π/2) ├─┤ P(θ[7]) ├───────────┤ X ├──■────────────»
     ├─────────────┤├─────────────┤ ├─────────┤           └───┘┌─┴─┐          »
q_4: ┤ R(x[4],π/2) ├┤ R(θ[8],π/2) ├─┤ P(θ[9]) ├────────────────┤ X ├──■───────»
     ├─────────────┤├─────────────┴┐├─────────┴┐               └───┘┌─┴─┐     »
q_5: ┤ R(x[5],π/2) ├┤ R(θ[10],π/2) ├┤ P(θ[11]) ├────────────────────┤ X ├──■──»
     ├─────────────┤├──────────────┤├──────────┤                    └───┘┌─┴─┐»
q_6: ┤ R(x[6],π/2) ├┤ R(θ[12],π/2) ├┤ P(θ[13]) ├─────────────────────────┤ X ├»
     ├─────────────┤├──────────────┤├──────────┤                         └───┘»
q_7: ┤ R(x[7],π/2) ├┤ R(θ[14],π/2) ├┤ P(θ[15]) ├──────────────────────────────»
     ├─────────────┤├──────────────┤├──────────┤                              »
q_8: ┤ R(x[8],π/2) ├┤ R(θ[16],π/2) ├┤ P(θ[17]) ├──────────────────────────────»
     └─────────────┘└──────────────┘└──────────┘                              »
«               ┌───┐
«q_0: ──────────┤ X ├
«               └─┬─┘
«q_1: ────────────┼──
«                 │  
«q_2: ────────────┼──
«                 │  
«q_3: ────────────┼──
«                 │  
«q_4: ────────────┼──
«                 │  
«q_5: ────────────┼──
«                 │  
«q_6: ──■─────────┼──
«     ┌─┴─┐       │  
«q_7: ┤ X ├──■────┼──
«     └───┘┌─┴─┐  │  
«q_8: ─────┤ X ├──■──
«          └───┘

In [8]:
observable = SparsePauliOp.from_list([("Z" + "I"*(n_qubits-1), 1.0)])

estimator = Estimator()

qnn = EstimatorQNN(
    circuit=qc,
    estimator=estimator,
    observables=observable,
    input_params=list(x_params),
    weight_params=list(theta_params),
)

No gradient function provided, creating a gradient function. If your Estimator requires transpilation, please provide a pass manager.


In [ ]:
torch.manual_seed(0)

model = TorchConnector(qnn)

loss_fn = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.05)

# Use a small batch for quick testing; increase for full training
n_train = 10000
xtr = torch.tensor(x_train_a[:n_train], dtype=torch.float32)
ytr = torch.tensor(y_train_s[:n_train], dtype=torch.float32).view(-1, 1)

for epoch in range(5):
    optimizer.zero_grad()
    yhat = model(xtr)
    loss = loss_fn(yhat, ytr)
    loss.backward()
    optimizer.step()
    print(f"epoch={epoch+1:3d}, loss={loss.item():.6f}")

epoch=  1, loss=0.989848
epoch=  2, loss=0.986963
epoch=  3, loss=0.983258
epoch=  4, loss=0.982979
epoch=  5, loss=0.981566
